# FourBi Dataset Evaluation Notebook

Notebook nay chay doc lap de danh gia checkpoint FourBi tren mot dataset. Sua `CHECKPOINT_PATH`, `DATASET_DIR`, sau do Run All.

Dataset mac dinh duoc ky vong co cau truc `imgs/` va `gt_imgs/`. Output gom anh da xu ly, `metrics_per_image.csv`, va `metrics_summary.csv`.


In [ ]:
from pathlib import Path
import math
import time
import csv

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# ========================= CONFIGURATION =========================
# Sua 2 duong dan nay, sau do Run All.
CHECKPOINT_PATH = Path('ckpts/3d22_HDIBCO18.pth')
DATASET_DIR = Path('path/to/dataset')  # folder dataset, thuong co DATASET_DIR/imgs va DATASET_DIR/gt_imgs

# Folder luu anh da xu ly va file metrics.
OUTPUT_DIR = Path('output_eval')

# Neu dataset khong theo cau truc imgs/gt_imgs, dat truc tiep 2 folder ben duoi.
# De None thi notebook tu tim DATASET_DIR/imgs va DATASET_DIR/gt_imgs.
IMAGE_DIR = None
GROUND_TRUTH_DIR = None

PATCH_SIZE = 512
BATCH_SIZE = 8
THRESHOLD = None  # None: lay threshold trong checkpoint; hoac dat vi du 0.5 / 0.0
MAX_DISPLAY = 5  # so anh hien thi o cuoi notebook; dat 0 de khong hien thi
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Dataset: {DATASET_DIR}')


## Model definition

Cell nay nhung kien truc FourBi de notebook khong can import cac file `.py` trong repo.


In [ ]:
def get_activation(kind='tanh'):
    if kind == 'tanh':
        return nn.Tanh()
    if kind == 'sigmoid':
        return nn.Sigmoid()
    if kind is False:
        return nn.Identity()
    raise ValueError(f'Unknown activation kind: {kind}')


class FourierUnit(nn.Module):
    def __init__(self, in_channels, out_channels, groups=1, spatial_scale_factor=None,
                 spatial_scale_mode='bilinear', spectral_pos_encoding=False,
                 ffc3d=False, fft_norm='ortho'):
        super().__init__()
        self.groups = groups
        self.conv_layer = nn.Conv2d(in_channels * 2 + (2 if spectral_pos_encoding else 0),
                                    out_channels * 2, kernel_size=1, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_channels * 2)
        self.relu = nn.ReLU(inplace=True)
        self.spatial_scale_factor = spatial_scale_factor
        self.spatial_scale_mode = spatial_scale_mode
        self.spectral_pos_encoding = spectral_pos_encoding
        self.ffc3d = ffc3d
        self.fft_norm = fft_norm

    def forward(self, x):
        batch = x.shape[0]
        if self.spatial_scale_factor is not None:
            original_size = x.shape[-2:]
            x = F.interpolate(x, scale_factor=self.spatial_scale_factor,
                              mode=self.spatial_scale_mode, align_corners=False)
        fft_dim = (-3, -2, -1) if self.ffc3d else (-2, -1)
        transformed = torch.fft.rfftn(x, dim=fft_dim, norm=self.fft_norm)
        transformed = torch.stack((transformed.real, transformed.imag), dim=-1)
        transformed = transformed.permute(0, 1, 4, 2, 3).contiguous()
        transformed = transformed.view((batch, -1) + transformed.size()[3:])
        if self.spectral_pos_encoding:
            height, width = transformed.shape[-2:]
            coords_v = torch.linspace(0, 1, height, device=x.device)[None, None, :, None].expand(batch, 1, height, width)
            coords_h = torch.linspace(0, 1, width, device=x.device)[None, None, None, :].expand(batch, 1, height, width)
            transformed = torch.cat((coords_v, coords_h, transformed), dim=1)
        transformed = self.relu(self.bn(self.conv_layer(transformed)))
        transformed = transformed.view((batch, -1, 2) + transformed.size()[2:]).permute(0, 1, 3, 4, 2).contiguous()
        transformed = torch.complex(transformed[..., 0], transformed[..., 1])
        output = torch.fft.irfftn(transformed, s=x.shape[-3:] if self.ffc3d else x.shape[-2:],
                                  dim=fft_dim, norm=self.fft_norm)
        if self.spatial_scale_factor is not None:
            output = F.interpolate(output, size=original_size, mode=self.spatial_scale_mode, align_corners=False)
        return output


class SpectralTransform(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, groups=1, enable_lfu=True, **fu_kwargs):
        super().__init__()
        self.enable_lfu = enable_lfu
        self.downsample = nn.AvgPool2d(2, stride=2) if stride == 2 else nn.Identity()
        self.conv1 = nn.Sequential(nn.Conv2d(in_channels, out_channels // 2, 1, groups=groups, bias=False),
                                   nn.BatchNorm2d(out_channels // 2), nn.ReLU(inplace=True))
        self.fu = FourierUnit(out_channels // 2, out_channels // 2, groups, **fu_kwargs)
        if enable_lfu:
            self.lfu = FourierUnit(out_channels // 2, out_channels // 2, groups)
        self.conv2 = nn.Conv2d(out_channels // 2, out_channels, 1, groups=groups, bias=False)

    def forward(self, x):
        x = self.conv1(self.downsample(x))
        output = self.fu(x)
        if self.enable_lfu:
            _, channels, height, _ = x.shape
            split_size = height // 2
            xs = torch.cat(torch.split(x[:, :channels // 4], split_size, dim=-2), dim=1).contiguous()
            xs = torch.cat(torch.split(xs, split_size, dim=-1), dim=1).contiguous()
            xs = self.lfu(xs).repeat(1, 1, 2, 2).contiguous()
        else:
            xs = 0
        return self.conv2(x + output + xs)


class FFC(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, ratio_gin, ratio_gout,
                 stride=1, padding=0, dilation=1, groups=1, bias=False, enable_lfu=True,
                 padding_type='reflect', gated=False, **spectral_kwargs):
        super().__init__()
        in_cg, out_cg = int(in_channels * ratio_gin), int(out_channels * ratio_gout)
        in_cl, out_cl = in_channels - in_cg, out_channels - out_cg
        self.ratio_gout = ratio_gout
        self.global_in_num = in_cg
        kwargs = dict(kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=bias, padding_mode=padding_type)
        self.convl2l = nn.Identity() if in_cl == 0 or out_cl == 0 else nn.Conv2d(in_cl, out_cl, **kwargs)
        self.convl2g = nn.Identity() if in_cl == 0 or out_cg == 0 else nn.Conv2d(in_cl, out_cg, **kwargs)
        self.convg2l = nn.Identity() if in_cg == 0 or out_cl == 0 else nn.Conv2d(in_cg, out_cl, **kwargs)
        self.convg2g = nn.Identity() if in_cg == 0 or out_cg == 0 else SpectralTransform(
            in_cg, out_cg, stride, 1 if groups == 1 else groups // 2, enable_lfu, **spectral_kwargs)
        self.gated = gated
        self.gate = nn.Identity() if in_cg == 0 or out_cl == 0 or not gated else nn.Conv2d(in_channels, 2, 1)

    def forward(self, x):
        x_l, x_g = x if type(x) is tuple else (x, 0)
        if self.gated:
            total = torch.cat([x_l, x_g], dim=1) if torch.is_tensor(x_g) else x_l
            g2l_gate, l2g_gate = torch.sigmoid(self.gate(total)).chunk(2, dim=1)
        else:
            g2l_gate = l2g_gate = 1
        out_l = self.convl2l(x_l) + self.convg2l(x_g) * g2l_gate if self.ratio_gout != 1 else 0
        out_g = self.convl2g(x_l) * l2g_gate + self.convg2g(x_g) if self.ratio_gout != 0 else 0
        return out_l, out_g


class FFC_BN_ACT(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, ratio_gin, ratio_gout,
                 stride=1, padding=0, dilation=1, groups=1, bias=False, norm_layer=nn.BatchNorm2d,
                 activation_layer=nn.Identity, padding_type='reflect', enable_lfu=True, **kwargs):
        super().__init__()
        self.ffc = FFC(in_channels, out_channels, kernel_size, ratio_gin, ratio_gout, stride, padding,
                       dilation, groups, bias, enable_lfu, padding_type=padding_type, **kwargs)
        global_channels = int(out_channels * ratio_gout)
        self.bn_l = nn.Identity() if ratio_gout == 1 else norm_layer(out_channels - global_channels)
        self.bn_g = nn.Identity() if ratio_gout == 0 else norm_layer(global_channels)
        self.act_l = nn.Identity() if ratio_gout == 1 else activation_layer(inplace=True)
        self.act_g = nn.Identity() if ratio_gout == 0 else activation_layer(inplace=True)

    def forward(self, x):
        x_l, x_g = self.ffc(x)
        return self.act_l(self.bn_l(x_l)), self.act_g(self.bn_g(x_g))


class FFCResnetBlock(nn.Module):
    def __init__(self, dim, padding_type, norm_layer, activation_layer=nn.ReLU, dilation=1, inline=False, **kwargs):
        super().__init__()
        self.conv1 = FFC_BN_ACT(dim, dim, 3, padding=dilation, dilation=dilation, norm_layer=norm_layer,
                                activation_layer=activation_layer, padding_type=padding_type, **kwargs)
        self.conv2 = FFC_BN_ACT(dim, dim, 3, padding=dilation, dilation=dilation, norm_layer=norm_layer,
                                activation_layer=activation_layer, padding_type=padding_type, **kwargs)
        self.inline = inline

    def forward(self, x):
        if self.inline:
            x_l, x_g = x[:, :-self.conv1.ffc.global_in_num], x[:, -self.conv1.ffc.global_in_num:]
        else:
            x_l, x_g = x if type(x) is tuple else (x, 0)
        id_l, id_g = x_l, x_g
        x_l, x_g = self.conv1((x_l, x_g))
        x_l, x_g = self.conv2((x_l, x_g))
        result = (id_l + x_l, id_g + x_g)
        return torch.cat(result, dim=1) if self.inline else result


class ConcatTupleLayer(nn.Module):
    def forward(self, x):
        return torch.cat(x, dim=1) if torch.is_tensor(x[1]) else x[0]


class Fourbi(nn.Module):
    def __init__(self, input_nc, output_nc, ngf=64, n_downsampling=3, n_blocks=3,
                 norm_layer=nn.BatchNorm2d, padding_type='reflect', activation_layer=nn.ReLU,
                 up_norm_layer=nn.BatchNorm2d, up_activation=nn.ReLU(True), init_conv_kwargs=None,
                 downsample_conv_kwargs=None, resnet_conv_kwargs=None, add_out_act=True,
                 max_features=1024, out_ffc=False, skip_connections='cat', unet_layers=2):
        super().__init__()
        init_conv_kwargs = init_conv_kwargs or {}
        downsample_conv_kwargs = downsample_conv_kwargs or {}
        resnet_conv_kwargs = resnet_conv_kwargs or {}
        conv3 = dict(kernel_size=3, stride=1, padding=1)
        self.reflect = nn.ReflectionPad2d(3)
        self.skip_connections = skip_connections
        down_channels = [ngf]
        layer = [FFC_BN_ACT(input_nc, ngf, 7, padding=0, norm_layer=norm_layer,
                            activation_layer=activation_layer, **init_conv_kwargs)]
        for _ in range(unet_layers):
            layer.append(FFC_BN_ACT(ngf, ngf, norm_layer=norm_layer, activation_layer=activation_layer,
                                    **init_conv_kwargs, **conv3))
        down_layers = [nn.Sequential(*layer)]
        for index in range(n_downsampling):
            mult = 2 ** index
            kwargs = dict(downsample_conv_kwargs)
            if index == n_downsampling - 1:
                kwargs['ratio_gout'] = resnet_conv_kwargs.get('ratio_gin', 0)
            channels = min(max_features, ngf * mult * 2)
            down_channels.append(channels)
            layer = [FFC_BN_ACT(min(max_features, ngf * mult), channels, 3, stride=2, padding=1,
                                norm_layer=norm_layer, activation_layer=activation_layer, **kwargs)]
            if index < n_downsampling - 1:
                for _ in range(unet_layers):
                    layer.append(FFC_BN_ACT(channels, channels, norm_layer=norm_layer,
                                            activation_layer=activation_layer, **kwargs, **conv3))
            down_layers.append(nn.Sequential(*layer))
        bottleneck = min(max_features, ngf * (2 ** n_downsampling))
        resnet_layers = [FFCResnetBlock(bottleneck, padding_type, norm_layer,
                                         activation_layer=activation_layer, **resnet_conv_kwargs)
                         for _ in range(n_blocks)] + [ConcatTupleLayer()]
        up_layers = []
        for index in range(n_downsampling):
            mult = 2 ** (n_downsampling - index)
            in_channels = min(max_features, ngf * mult)
            in_channels += down_channels.pop() if skip_connections == 'cat' else 0
            channels = min(max_features, int(ngf * mult / 2))
            layer = [nn.ConvTranspose2d(in_channels, channels, 3, stride=2, padding=1, output_padding=1),
                     up_norm_layer(channels), up_activation]
            for _ in range(unet_layers):
                layer.extend([nn.Conv2d(channels, channels, **conv3), nn.BatchNorm2d(channels), nn.ReLU()])
            up_layers.append(nn.Sequential(*layer))
        if out_ffc:
            raise NotImplementedError
        out_channels = ngf + (down_channels.pop() if skip_connections == 'cat' else 0)
        temporary_out = output_nc if unet_layers == 0 else out_channels
        layer = [nn.ReflectionPad2d(3), nn.Conv2d(out_channels, temporary_out, 7)]
        for index in range(unet_layers):
            temporary_out = output_nc if index == unet_layers - 1 else out_channels
            layer.extend([nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Conv2d(out_channels, temporary_out, 3, padding=1)])
        up_layers.append(nn.Sequential(*layer))
        self.final_act = get_activation('tanh' if add_out_act is True else add_out_act)
        self.down_sampling_layers = nn.Sequential(*down_layers)
        self.resnet_layers = nn.Sequential(*resnet_layers)
        self.up_sampling_layers = nn.Sequential(*up_layers)

    def forward(self, x):
        x = self.reflect(x)
        skip_outputs = []
        for layer in self.down_sampling_layers:
            x = layer(x)
            if self.skip_connections != 'none':
                parts = x if torch.is_tensor(x[1]) else (x[0],)
                skip_outputs.append(torch.cat(parts, dim=1))
        x = self.resnet_layers(x)
        for layer in self.up_sampling_layers:
            if self.skip_connections != 'none':
                skip = skip_outputs.pop()
                x = torch.cat([x, skip], dim=1) if self.skip_connections == 'cat' else x + skip
            x = layer(x)
        return self.final_act(x)

## Load checkpoint


In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint: {CHECKPOINT_PATH}')

# FourBi checkpoints contain config/training state, so weights_only=False is required.
# Chi su dung checkpoint tu nguon tin cay.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
config = checkpoint['config']
model = Fourbi(
    input_nc=config['input_channels'],
    output_nc=config['output_channels'],
    n_downsampling=config['n_downsampling'],
    init_conv_kwargs=config['init_conv_kwargs'],
    downsample_conv_kwargs=config['down_sample_conv_kwargs'],
    resnet_conv_kwargs=config['resnet_conv_kwargs'],
    n_blocks=config['n_blocks'],
    unet_layers=config['unet_layers'],
).to(DEVICE)
model.load_state_dict(checkpoint['model'], strict=True)
model.eval()
threshold = config.get('threshold', 0.5) if THRESHOLD is None else THRESHOLD

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Loaded checkpoint: {CHECKPOINT_PATH.name}')
print(f'Patch size: {PATCH_SIZE}, batch size: {BATCH_SIZE}, threshold: {threshold}')


## Inference and metric utilities


In [ ]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}

def resolve_image_and_gt_dirs(dataset_dir, image_dir=None, gt_dir=None):
    dataset_dir = Path(dataset_dir)
    image_dir = Path(image_dir) if image_dir is not None else (dataset_dir / 'imgs' if (dataset_dir / 'imgs').is_dir() else dataset_dir)
    gt_dir = Path(gt_dir) if gt_dir is not None else (dataset_dir / 'gt_imgs')
    if not image_dir.is_dir():
        raise FileNotFoundError(f'Khong tim thay folder anh: {image_dir}')
    if not gt_dir.is_dir():
        raise FileNotFoundError(f'Khong tim thay folder ground truth: {gt_dir}')
    return image_dir, gt_dir


def find_images(image_dir):
    paths = sorted(p for p in Path(image_dir).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise FileNotFoundError(f'Khong tim thay anh trong: {image_dir}')
    return paths


def find_ground_truth(image_path, gt_dir):
    gt_dir = Path(gt_dir)
    exact_match = gt_dir / image_path.name
    if exact_match.exists():
        return exact_match
    for suffix in IMAGE_EXTENSIONS:
        candidate = gt_dir / f'{image_path.stem}{suffix}'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Khong tim thay ground truth cho {image_path.name} trong {gt_dir}')


def image_to_tensor(image):
    array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


def make_patches(image, patch_size):
    stride = patch_size // 2
    tensor = image_to_tensor(image).unsqueeze(0)
    _, _, height, width = tensor.shape
    padding_bottom = ((height // patch_size) + 1) * patch_size - height
    padding_right = ((width // patch_size) + 1) * patch_size - width
    tensor = F.pad(tensor, (0, padding_right, 0, padding_bottom), value=1.0)
    unfolded = tensor.unfold(2, patch_size, stride).unfold(3, patch_size, stride)
    n_rows, n_cols = unfolded.shape[2], unfolded.shape[3]
    patches = unfolded.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, patch_size, patch_size)
    return patches, n_rows, n_cols, stride, (height, width)


def stitch_predictions(predictions, n_rows, n_cols, stride, original_size):
    patch_size = predictions.shape[-1]
    padded_height = patch_size + (n_rows - 1) * stride
    padded_width = patch_size + (n_cols - 1) * stride
    patches = predictions.reshape(n_rows, n_cols, 1, patch_size, patch_size)
    canvas = torch.zeros(1, padded_height, padded_width)
    x_steps = [x + stride // 2 for x in range(0, padded_height, stride)]
    y_steps = [y + stride // 2 for y in range(0, padded_width, stride)]
    x_steps[0], x_steps[-1] = 0, padded_height
    y_steps[0], y_steps[-1] = 0, padded_width
    for row in range(n_rows):
        for col in range(n_cols):
            x1, x2 = x_steps[row], x_steps[row + 1]
            y1, y2 = y_steps[col], y_steps[col + 1]
            px1, px2 = x1 - row * stride, x2 - row * stride
            py1, py2 = y1 - col * stride, y2 - col * stride
            canvas[:, x1:x2, y1:y2] = patches[row, col, :, px1:px2, py1:py2]
    height, width = original_size
    return canvas[:, :height, :width]


@torch.inference_mode()
def binarize_image(image_path):
    source = Image.open(image_path).convert('RGB')
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    patches, n_rows, n_cols, stride, size = make_patches(source, PATCH_SIZE)
    predictions = []
    for batch in torch.split(patches, BATCH_SIZE):
        predictions.append(model(batch.to(DEVICE)).cpu())
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    prediction = stitch_predictions(torch.cat(predictions), n_rows, n_cols, stride, size)
    binary = (prediction > threshold).to(torch.uint8).squeeze(0).numpy() * 255
    inference_time = time.perf_counter() - start
    return source, Image.fromarray(binary, mode='L'), inference_time


def to_binary_array(image):
    arr = np.asarray(image.convert('L'), dtype=np.uint8)
    return (arr > 127).astype(np.uint8)


def calculate_psnr(pred, gt):
    pred = to_binary_array(pred).astype(np.float32)
    gt = to_binary_array(gt).astype(np.float32)
    mse = np.mean((pred - gt) ** 2)
    return float('inf') if mse == 0 else float(10 * np.log10(1.0 / mse))


def calculate_fm(pred, gt):
    pred_fg = 1 - to_binary_array(pred)
    gt_fg = 1 - to_binary_array(gt)
    tp = np.logical_and(pred_fg == 1, gt_fg == 1).sum()
    fp = np.logical_and(pred_fg == 1, gt_fg == 0).sum()
    fn = np.logical_and(pred_fg == 0, gt_fg == 1).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    fm = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return fm * 100.0, precision * 100.0, recall * 100.0


def skeletonize_binary(mask):
    # mask: foreground/text=True. Uses skimage when available; falls back to the original mask.
    try:
        from skimage.morphology import skeletonize
        return skeletonize(mask.astype(bool))
    except Exception:
        return mask.astype(bool)


def calculate_fps(pred, gt):
    pred_fg = (1 - to_binary_array(pred)).astype(bool)
    gt_fg = (1 - to_binary_array(gt)).astype(bool)
    gt_skeleton = skeletonize_binary(gt_fg)
    tp_precision = np.logical_and(pred_fg, gt_fg).sum()
    fp_precision = np.logical_and(pred_fg, ~gt_fg).sum()
    tp_recall = np.logical_and(pred_fg, gt_skeleton).sum()
    fn_recall = np.logical_and(~pred_fg, gt_skeleton).sum()
    precision = tp_precision / (tp_precision + fp_precision) if (tp_precision + fp_precision) else 0.0
    pseudo_recall = tp_recall / (tp_recall + fn_recall) if (tp_recall + fn_recall) else 0.0
    fps = 2 * precision * pseudo_recall / (precision + pseudo_recall) if (precision + pseudo_recall) else 0.0
    return fps * 100.0


def count_non_uniform_8x8_blocks(gt_binary):
    h, w = gt_binary.shape
    count = 0
    for y in range(0, h, 8):
        for x in range(0, w, 8):
            block = gt_binary[y:min(y + 8, h), x:min(x + 8, w)]
            if block.size and block.min() != block.max():
                count += 1
    return max(count, 1)


def calculate_drd(pred, gt):
    pred_bin = to_binary_array(pred).astype(np.float32)
    gt_bin = to_binary_array(gt).astype(np.float32)
    if pred_bin.shape != gt_bin.shape:
        raise ValueError(f'Kich thuoc output va ground truth khong trung: {pred_bin.shape} vs {gt_bin.shape}')

    weights = np.zeros((5, 5), dtype=np.float32)
    for i in range(5):
        for j in range(5):
            if i == 2 and j == 2:
                continue
            weights[i, j] = 1.0 / math.sqrt((i - 2) ** 2 + (j - 2) ** 2)
    weights /= weights.sum()

    diff_y, diff_x = np.where(pred_bin != gt_bin)
    padded_gt = np.pad(gt_bin, 2, mode='edge')
    drd_sum = 0.0
    h, w = gt_bin.shape
    for y, x in zip(diff_y, diff_x):
        if y < 2 or y >= h - 2 or x < 2 or x >= w - 2:
            continue
        patch = padded_gt[y:y + 5, x:x + 5]
        drd_sum += np.sum(np.abs(patch - pred_bin[y, x]) * weights)
    return drd_sum / count_non_uniform_8x8_blocks(gt_bin)


def visualize_results(results, max_display=5):
    if not max_display:
        return
    visible = results[:max_display]
    figure, axes = plt.subplots(len(visible), 3, figsize=(15, 4 * len(visible)), squeeze=False)
    for row, item in enumerate(visible):
        axes[row, 0].imshow(item['source'])
        axes[row, 0].set_title(f"Input: {item['name']}")
        axes[row, 1].imshow(item['prediction'], cmap='gray', vmin=0, vmax=255)
        axes[row, 1].set_title('Prediction')
        axes[row, 2].imshow(item['ground_truth'], cmap='gray', vmin=0, vmax=255)
        axes[row, 2].set_title('Ground truth')
        for col in range(3):
            axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()


## Run evaluation


In [ ]:
image_dir, gt_dir = resolve_image_and_gt_dirs(DATASET_DIR, IMAGE_DIR, GROUND_TRUTH_DIR)
image_paths = find_images(image_dir)
print(f'Found {len(image_paths)} image(s)')
print(f'Image dir: {image_dir}')
print(f'GT dir: {gt_dir}')
print(f'Output dir: {OUTPUT_DIR}')

rows = []
visual_rows = []
total_inference_time = 0.0
wall_start = time.perf_counter()

for index, image_path in enumerate(image_paths, start=1):
    gt_path = find_ground_truth(image_path, gt_dir)
    source, prediction, inference_time = binarize_image(image_path)
    ground_truth = Image.open(gt_path).convert('L')
    if prediction.size != ground_truth.size:
        raise ValueError(f'{image_path.name}: output size {prediction.size} != ground truth size {ground_truth.size}')

    output_path = OUTPUT_DIR / f'{image_path.stem}.png'
    prediction.save(output_path)

    psnr = calculate_psnr(prediction, ground_truth)
    fm, precision, recall = calculate_fm(prediction, ground_truth)
    fps = calculate_fps(prediction, ground_truth)
    drd = calculate_drd(prediction, ground_truth)
    total_inference_time += inference_time

    row = {
        'image': image_path.name,
        'output': str(output_path),
        'psnr': psnr,
        'fm': fm,
        'fps': fps,
        'drd': drd,
        'precision': precision,
        'recall': recall,
        'inference_time_sec': inference_time,
    }
    rows.append(row)
    if len(visual_rows) < max(MAX_DISPLAY, 0):
        visual_rows.append({'name': image_path.name, 'source': source, 'prediction': prediction, 'ground_truth': ground_truth})

    print(f"[{index}/{len(image_paths)}] {image_path.name} | PSNR={psnr:.4f} FM={fm:.4f} Fps={fps:.4f} DRD={drd:.4f} time={inference_time:.4f}s")

wall_time = time.perf_counter() - wall_start
finite_psnr = [r['psnr'] for r in rows if np.isfinite(r['psnr'])]
summary = {
    'num_images': len(rows),
    'mean_psnr': float(np.mean(finite_psnr)) if finite_psnr else float('inf'),
    'mean_fm': float(np.mean([r['fm'] for r in rows])),
    'mean_fps': float(np.mean([r['fps'] for r in rows])),
    'mean_drd': float(np.mean([r['drd'] for r in rows])),
    'total_inference_time_sec': total_inference_time,
    'avg_inference_time_sec': total_inference_time / len(rows),
    'throughput_images_per_sec': len(rows) / total_inference_time if total_inference_time > 0 else float('inf'),
    'wall_time_sec': wall_time,
}

metrics_csv = OUTPUT_DIR / 'metrics_per_image.csv'
summary_csv = OUTPUT_DIR / 'metrics_summary.csv'
with metrics_csv.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
with summary_csv.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary.keys()))
    writer.writeheader()
    writer.writerow(summary)

print('\n===== SUMMARY =====')
for key, value in summary.items():
    if isinstance(value, float):
        print(f'{key}: {value:.6f}')
    else:
        print(f'{key}: {value}')
print(f'\nSaved binarized images to: {OUTPUT_DIR}')
print(f'Saved per-image metrics to: {metrics_csv}')
print(f'Saved summary metrics to: {summary_csv}')

visualize_results(visual_rows, MAX_DISPLAY)
